# Featuring Enginnering Using Scikit-Learn

### Do You Want to Build a Snowman?

Let's prep some data for a model to predict the height of snowman.

<img src = "../assets/olaf.jpeg">

#### Load Packages

In [ ]:
# data analysis stack
import numpy as np
import pandas as pd

# data visualization stack
import matplotlib.pyplot as plt

%matplotlib inline
import seaborn as sns

sns.set_style("whitegrid")

# machine-learning stack
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    RobustScaler,
    MinMaxScaler,
    KBinsDiscretizer,
    PolynomialFeatures,
    FunctionTransformer,
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# miscellaneous
import warnings

warnings.filterwarnings("ignore")

#### Load Data

In [ ]:
data = {
    "temp": [-3, 5, 0, 7, 3, -1, 1, None, -6, 3, 0, -1, None, -2],
    "lunch": [
        "soup",
        "sandwich",
        "soup",
        "burger",
        "sandwich",
        "soup",
        "cereal",
        "salad",
        "sandwich",
        "burger",
        "soup",
        "cereal",
        "burger",
        "soup",
    ],
    "dinner": [
        "pizza",
        "pizza",
        "noodles",
        None,
        "fishsticks",
        "pizza",
        None,
        "fishsticks",
        "noodles",
        "pizza",
        None,
        "pizza",
        "fishsticks",
        "pizza",
    ],
    "precipitation": [
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "no",
    ],
    "height_snowman_cm": [100, 0, 75, 0, 20, 25, 0, 35, 170, 0, 85, 85, 45, 0],
}

df_train = pd.DataFrame(data=data)
df_train

#### Exercise
Transform the data above using scikit-learn tools in a way that is suitable to be used for modeling
+ **Separate the DataFrame `df_train` into `X_train` and `y_train`** 
   +  Our target variable is `height_snowman_cm`
+ **Preprocess `X_train`**:
  + Identify which variables are **binary**, **categorical** and  **numeric**
  + Check which variables have **missing values**
    + **Impute missing value** as needed using appropriate strategy
  + Determine if categorical variables have **non-numeric values**
    + **Encode categorical variables** using techniques such as one-hot encoding
  + Determine if numeric variables are on different scales
    + **Scale numeric variables**
+ **Create `X_train_fe`**:
    + Once the preprocessing steps are completed, compile the transformed columns into a new DataFrame called `X_train_fe`. 


1. **Separate the DataFrame `df_train` into `X_train` and `y_train`**

In [ ]:
target = "height_snowman_cm"

In [ ]:
# Feature matrix
X_train = df_train.drop(target, axis=1)
y_train = df_train[target]

In [ ]:
X_train.columns

2.1 **Identify which variables are binary, categorical and  numeric**

| Column Name | Type     |
|----------|----------------|
| temp       |numeric   |
| lunch        | categorical nominal|
| dinner        | categorical nominal|
| precipitation        | categorical binary|


2.2 **Check which variables have missing values**

In [ ]:
X_train.isna().sum()

In [ ]:
sns.heatmap(data=X_train.isna());

The variables `temp` and `dinner` contain missing values at observations 7 and 12 for `temp`, and at observations 3, 6, and 10 for `dinner`, respectively.

2.3 **Impute missing values**

**Numeric Variable `temp`**

We can fill the missing values by interpolation 

In [ ]:
def interpolate(X, column):
    X_copy = X.copy()
    X_copy[f"{column}_interpoleted"] = X_copy[[column]].interpolate()
    return X_copy[[f"{column}_interpoleted"]]

In [ ]:
from sklearn.preprocessing import FunctionTransformer

In [ ]:
interpolater = FunctionTransformer(interpolate, kw_args={"column": "temp"})

In [ ]:
interpolater.fit(X_train)

In [ ]:
temp_interpolated_train = interpolater.transform(X_train)
temp_interpolated_train

**Nominal Variable `dinner`**

We can fill the missing value by using as strategy the most frequent category

In [ ]:
dinner_imputer = SimpleImputer(
    strategy="most_frequent",
).set_output(transform="pandas")

In [ ]:
dinner_imputer.fit(X_train[["dinner"]])

In [ ]:
dinner_imputed_train = dinner_imputer.transform(X_train[["dinner"]])
dinner_imputed_train

2.4 **Encode Categorical variables**

**`lunch` and `precipitation`**

In [ ]:
subset_categoric = ["lunch", "precipitation"]

In [ ]:
ohe_encoder = OneHotEncoder(sparse_output=False, drop="first").set_output(
    transform="pandas"
)

In [ ]:
ohe_encoder.fit(X_train[subset_categoric])

In [ ]:
lunch_prep_encoded_train = ohe_encoder.transform(X_train[subset_categoric])
lunch_prep_encoded_train

**`imputed dinner`**

In [ ]:
dinner_encoder = OneHotEncoder(sparse_output=False, drop="first").set_output(
    transform="pandas"
)

In [ ]:
dinner_encoder.fit(dinner_imputed_train[["dinner"]])

| Column Name           | Type                                      | Value                                                                 |
|----------------|-------------------------------------------|-----------------------------------------------------------------------|
| temp | numeric                               | <class 'sklearn.compose._column_transformer.ColumnTransformer'>       |
| LinearRegression  | abc.ABCMeta                               | <class 'sklearn.linear_model._base.LinearRegression'>                 |
| Pipeline          | abc.ABCMeta                               | <class 'sklearn.pipeline.Pipeline'>                                   |
| X_train           | pandas.core.frame.DataFrame              | DataFrame with columns: temp, lunch, dinner, precipitation             |
| data              | dict                                      | {'temp': [-3, 5, 0, 7, 3, -1, 1, None, -6, 3, 0, -1, None, -2], ...}  |
| df_train          | pandas.core.frame.DataFrame              | DataFrame with columns: temp, lunch, dinner, precipitation, heigth_snowman_cm |
| target            | str                                       | 'heigth_snowman_cm'                                                   |
| y_train           | pandas.core.series.Series                | Series with target variable values                                    |

In [ ]:
dinner_encoded_train = dinner_encoder.transform(dinner_imputed_train[["dinner"]])
dinner_encoded_train

3. **Create X_train_fe**

In [ ]:
transformed_dataframes = [
    temp_interpolated_train,
    dinner_encoded_train,
    lunch_prep_encoded_train,
]

In [ ]:
X_train_fe = pd.concat(transformed_dataframes, axis=1)
X_train_fe